<a href="https://colab.research.google.com/github/heisenberg304/Housing-Prices-prediction/blob/develop/Data%20Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile data_housing.py
import pandas as pd

#cargar datos
data_original = pd.read_csv("/content/drive/MyDrive/house_prices_advanced/House_Prices_train.csv")

Writing data_housing.py


#FILLING DATASET TRAIN

In [ ]:
from data_housing import data_original
import pandas as pd


data = data_original.copy()
#revisar si hay valores faltantes con:
missing = data.isnull().sum().sort_values()
missing = missing[missing > 0]


#manejo de nan de basement
nan_bsmt = ['BsmtFinType1', 'BsmtQual', 'BsmtCond', 'BsmtFinType2', 'BsmtExposure']
data['BsmtFinType2'].value_counts()
data.loc[
    data["BsmtFinType2"].isna(),
    nan_bsmt
]
data.loc[948, 'BsmtExposure'] = 'No'
data.loc[332, 'BsmtFinType2'] = 'Unf'
data[nan_bsmt] = data[nan_bsmt].fillna('NoBsmt')


#manejo de nan de garage
nan_garage = ['GarageQual', 'GarageFinish', 'GarageType', 'GarageCond']
data.loc[
    data["GarageQual"].isna(),
    ['GarageQual', 'GarageYrBlt', 'GarageFinish', 'GarageType', 'GarageCond']
]
data[nan_garage] = data[nan_garage].fillna('NoGarage')
data['GarageYrBlt'] = data['GarageYrBlt'].fillna(0)

#manejo de nan en electrical
data['Electrical'].fillna(data['Electrical'].mode()[0], inplace = True)

#manejo de nan en MasVnrArea
data.loc[
    data["MasVnrArea"].isna(),
    ['MasVnrArea', 'MasVnrType']
]
data['MasVnrArea'] = data['MasVnrArea'].fillna(0)

#manejo de nan de PoolQC
pd.crosstab(
    data["PoolQC"].isna(),
    data["PoolArea"] == 0
)
data['PoolQC'] = data['PoolQC'].fillna('NoPool')

#manejo de nan de Fence
data["Fence"] = data["Fence"].fillna("NoFence")

#manejo de nan de Alley
data["Alley"] = data["Alley"].fillna("NoAlley")

#manejo de nan de LotFrontage
data['MedianVecindario'] = data.groupby("Neighborhood")["LotFrontage"].transform("median")
data.loc[
     data['LotFrontage'].isnull(),
      ['LotFrontage', 'MedianVecindario']
]
data["LotFrontage"] = data["LotFrontage"].fillna(data["MedianVecindario"])
data = data.drop("MedianVecindario", axis=1)


#manejo de Nan de FireplaceQu
data.loc[
    data['FireplaceQu'].isna(),
    'Fireplaces'
].value_counts()
data["FireplaceQu"] = data["FireplaceQu"].fillna("NoFrPlcQu")

#Manejo de nan de MasVnrType
data['MasVnrType'].value_counts()
data.loc[
    data['MasVnrType'].isna(),
    'MasVnrArea'
].value_counts()

data.loc[624, 'MasVnrType'] = 'BrkFace'
data.loc[1300, 'MasVnrType'] = 'BrkFace'
data.loc[1334, 'MasVnrType'] = 'BrkFace'

data['MasVnrType'] = data['MasVnrType'].fillna('NoMasVnrType')

#Manejo de nan de MiscFeature
data['MiscFeature'] = data['MiscFeature'].fillna('NoMiscFeature')

#eliminar id
data = data.drop("Id", axis=1)

missing = data.isnull().sum().sort_values()
missing[missing > 0]

#guardar csv
data.to_csv("HP_train_filled.csv", index=False)

/tmp/ipykernel_4544/4200846218.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Electrical'].fillna(data['Electrical'].mode()[0], inplace = True)


#Analizar cada caso de variable con NaNs

Estoy rellenando para los modelos mas basicos que no soportan NaNs, despues probare los modelos que soportan NaNs con el dataset rellenado y luego con NaNs para ver si hay una diferencia en el rendimiento

---

##Manejo de NaN en las variables restantes

Electrical (1): Sistema eléctrico de la vivienda.

Se identificó un único valor faltante en esta variable categórica. Debido a que la categoría dominante representaba ampliamente la distribución de la variable y el porcentaje de datos faltantes era insignificante, el valor fue imputado utilizando la moda de la variable.


MasVnrArea (8): Área del revestimiento decorativo de mampostería.

Se observó que los valores faltantes coincidían con observaciones sin información en MasVnrType. Dado que estas viviendas no presentan revestimiento decorativo, los valores faltantes fueron reemplazados por 0, representando ausencia de área de revestimiento.


PoolQC (1453): Calidad de la piscina.

Se verificó mediante un análisis cruzado con PoolArea que todas las viviendas con PoolQC faltante presentaban un área de piscina igual a 0. Esto indica que los valores faltantes representan ausencia de piscina y no pérdida de información. Por esta razón se creó la categoría "NoPool".


Fence (1179): Calidad o tipo de cerca.

La variable no dispone de una característica asociada que permita validar directamente su ausencia. Sin embargo, dada la naturaleza de la variable, se consideró que los valores faltantes representan viviendas sin cerca. Por ello se creó la categoría "NoFence".


Alley (1369): Tipo de callejón de acceso.

Los valores faltantes fueron interpretados como ausencia de callejón de acceso. Para conservar esta información se creó la categoría "NoAlley".


MiscFeature (1406): Característica adicional no común.

Los valores faltantes indican que la vivienda no posee ninguna característica especial adicional registrada. Por tanto, se creó la categoría "NoMiscFeature".


FireplaceQu (690): Calidad de la chimenea.

Se verificó que las viviendas con valores faltantes en FireplaceQu presentaban 0 chimeneas registradas en la variable Fireplaces. Por tanto, los valores faltantes representan ausencia de chimenea y no pérdida de información. Se creó la categoría "NoFrPlcQu".


LotFrontage (259): Longitud frontal del terreno conectada a la calle.

Al tratarse de una variable numérica, no era apropiado utilizar una categoría especial para representar los valores faltantes. Se observó además que la variable presenta diferencias significativas entre vecindarios, por lo que una imputación global podría introducir sesgo.

Para cada observación faltante se calculó la mediana de LotFrontage dentro de su respectivo vecindario (Neighborhood), utilizando posteriormente dicho valor para realizar la imputación. Esta estrategia permite preservar mejor las características propias de cada zona residencial.

---
###Manejo de NaN en las variables faltantes relacionadas con basement

Todas estos valores faltantes seran rellenados con esta nueva categorias 'NoBsmt', habran 2 casos especiales explicados a continuacion.
* BsmtFinType1	(37): Área terminada del sótano principal.
* BsmtQual	(37): Calidad del sótano.
* BsmtCond	(37): Condición del sótano.
* BsmtFinType2	(38): La observación 332 presenta un valor faltante en BsmtFinType2.
Dado que la vivienda sí posee sótano terminado (BsmtFinSF2 > 0),
el valor faltante no representa ausencia de sótano.

Al tratarse de un único registro y no existir información suficiente
para inferir la categoría exacta, se imputó utilizando la categoría
más frecuente (Unf).
* BsmtExposure	(38): Porque la vivienda posee sótano
(TotalBsmtSF = 936).
Por tanto el NaN no representa ausencia de sótano.

Analicé la distribución de BsmtExposure y observé que la categoría "No" era ampliamente dominante.
Dado que se trata de un único registro anómalo, imputar "No" preserva la consistencia del dataset sin introducir una categoría incorrecta.

---
###Manejo de NaN en las variables faltantes relacionadas con Garage

* GarageQual	(81): Calidad del garage.
* GarageFinish	(81): Nivel de acabado interior del garage.
* GarageType	(81): Tipo de garage.
* GarageCond	(81): Condición del garage.


Todos los valores faltantes asociados al garage fueron analizados en conjunto. Se observó que los valores faltantes coincidían en las mismas filas para las variables categóricas relacionadas con el garage, lo que indica que dichas viviendas no disponen de esta característica.

Dado que los valores faltantes representan ausencia de garage y no pérdida de información, se creó la categoría "NoGarage" para conservar esta información de manera explícita dentro del dataset.

* GarageYrBlt	(81): Año de construcción del garage.

Al tratarse de una variable numérica, no era apropiado imputarla con una categoría textual. Debido a que las viviendas sin garage no poseen un año de construcción asociado a esta característica, los valores faltantes fueron reemplazados por 0, utilizándolo como valor indicador de ausencia de garage.

##Features a eliminar
* Id: No aporta informacion relevante

Identificador único.

No tiene relación causal con el precio.

Puede inducir ruido.

Se elimina.


#FILLING DATASET TEST

In [ ]:
import pandas as pd

test_original = pd.read_csv('/content/drive/MyDrive/house_prices_advanced/House_Prices_test.csv')

test = test_original.copy()


#manejo de nan de PoolQC
pd.crosstab(
    test["PoolQC"].isna(),
    test["PoolArea"] == 0
)
test['PoolQC'] = test['PoolQC'].fillna('NoPool')


#manejo de nan de Fence
test["Fence"] = test["Fence"].fillna("NoFence")


#manejo de nan de Alley
test["Alley"] = test["Alley"].fillna("NoAlley")


#manejo de Nan de FireplaceQu
test.loc[
    test['FireplaceQu'].isna(),
    'Fireplaces'
].value_counts()
test["FireplaceQu"] = test["FireplaceQu"].fillna("NoFrPlcQu")


#Manejo de nan de MiscFeature
test['MiscFeature'] = test['MiscFeature'].fillna('NoMiscFeature')


#eliminar id
test = test.drop("Id", axis=1)


#manejo de basement
nan_bsmt = ['BsmtFinType1', 'BsmtFinType2', 'BsmtExposure', 'BsmtQual', 'BsmtCond']
int_bsmt = ['TotalBsmtSF', 'BsmtFinSF1', 'BsmtUnfSF', 'BsmtFinSF2', 'BsmtFullBath', 'BsmtHalfBath', ]
test[int_bsmt] = test[int_bsmt].fillna(0)
test['BsmtCond'].value_counts()
test.loc[580, 'BsmtCond'] = 'TA'
test.loc[725, 'BsmtCond'] = 'TA'
test.loc[1064, 'BsmtCond'] = 'TA'
test[nan_bsmt] = test[nan_bsmt].fillna('NoBsmt')
test.loc[
    test["BsmtCond"].isna(),
    ['TotalBsmtSF', 'BsmtFinSF1', 'BsmtUnfSF', 'BsmtFinSF2', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure', 'BsmtQual', 'BsmtCond']
]


#Manejo de nan de MasVnrType
test.loc[
    test['MasVnrType'].isna(),
    ['MasVnrArea', 'MasVnrType']
] #no hay relacion
test['MasVnrType'].value_counts()
test['MasVnrType'] = test['MasVnrType'].fillna(test['MasVnrType'].mode()[0])


#manejo de SaleType
test['SaleType'].value_counts()
test['SaleType'] = test['SaleType'].fillna(test['SaleType'].mode()[0])


#manejo de Functional
test['Functional'].value_counts()
test['Functional'] = test['Functional'].fillna(test['Functional'].mode()[0])


#manejo de Utilities
test['Utilities'].value_counts()
test['Utilities'] = test['Utilities'].fillna(test['Utilities'].mode()[0])


#manejo de MSZoning
test['MSZoning'].value_counts()
test['MSZoning'] = test['MSZoning'].fillna(test['MSZoning'].mode()[0])


#manejo de KitchenQual
test['KitchenQual'].value_counts()
test['KitchenQual'] = test['KitchenQual'].fillna(test['KitchenQual'].mode()[0])


#manejo de Exterior2nd Exterior1st
nan_exter = ['Exterior1st', 'Exterior2nd']
test.loc[
    test['Exterior1st'].isna(),
    nan_exter
] #son el mismo
test['Exterior1st'].value_counts()
test[nan_exter] = test[nan_exter].fillna('VinylSd')


#manejo de garage
test['GarageYrBlt'] = test['GarageYrBlt'].fillna(0)
test.loc[1116, 'GarageArea'] = (test['GarageArea'].median())
test.loc[1116, 'GarageCars'] = (test['GarageCars'].mode()[0])
test.loc[1116, 'GarageCond'] = 'TA'
test.loc[1116, 'GarageQual'] = 'TA'
test.loc[666, 'GarageQual'] = 'TA'
test.loc[666, 'GarageQual'] = 'TA'
test.loc[666, 'GarageFinish'] = (test['GarageFinish'].mode()[0])
test.loc[1116, 'GarageFinish'] = (test['GarageFinish'].mode()[0])

nan_garage = ['GarageQual', 'GarageCond', 'GarageType', 'GarageFinish']
test[nan_garage] = test[nan_garage].fillna('NoGarage')
resultado = test.loc[
    test['GarageQual'].isna(),
    ['GarageYrBlt', 'GarageQual', 'GarageCond', 'GarageType', 'GarageCars', 'GarageArea', 'GarageFinish']
]


#manejo de nan en MasVnrArea
import matplotlib.pyplot as plt
test.loc[
    test["MasVnrArea"].isna(),
    ['MasVnrArea', 'MasVnrType']
] #todos son BrkFace
BrkFace = test.loc[test['MasVnrType'] == 'BrkFace']
#plt.hist(BrkFace['MasVnrArea'])
#plt.show() #casi todos son 0
test['MasVnrArea'] = test['MasVnrArea'].fillna(0)


#manejo de nan de LotFrontage
test['MedianVecindario'] = test.groupby("Neighborhood")["LotFrontage"].transform("median")
test.loc[
     test['LotFrontage'].isnull(),
      ['LotFrontage', 'MedianVecindario']
]
test["LotFrontage"] = test["LotFrontage"].fillna(test["MedianVecindario"])
test = test.drop("MedianVecindario", axis=1)

missing = test.isnull().sum().sort_values()
missing[missing > 0]

#guardar csv
test.to_csv("HP_test_filled.csv", index=False)

#ONE - HOT ENCODING

In [3]:
import pandas as pd

#cargar datos
# data train no filled
train_not_filled = pd.read_csv("/content/drive/MyDrive/house_prices_advanced/House_Prices_train.csv")
train_not_filled = train_not_filled.drop("Id", axis=1)

# data train filled
train_filled = pd.read_csv("/content/drive/MyDrive/house_prices_advanced/HP_train_filled.csv")

# data test no filled
test_not_filled = pd.read_csv("/content/drive/MyDrive/house_prices_advanced/House_Prices_test.csv")
test_not_filled = test_not_filled.drop("Id", axis=1)

# data test filled
test_filled = pd.read_csv("/content/drive/MyDrive/house_prices_advanced/HP_test_filled.csv")



#ONE-HOT ENCODING
object_cols = train_filled.select_dtypes(include=['object']).columns
#
train_not_filled = pd.get_dummies(train_not_filled, columns=list(object_cols), drop_first=True)  # 245 columns
test_not_filled = pd.get_dummies(test_not_filled, columns=list(object_cols), drop_first=True)    # 226 columns

#
train_filled = pd.get_dummies(train_filled, columns=list(object_cols), drop_first=True)  # 260 columns
test_filled = pd.get_dummies(test_filled, columns=list(object_cols), drop_first=True)    # 240 columns


def searching_miss_col(train, test):
  train = train.drop("SalePrice", axis=1)
  train_columns = train.columns.to_list()
  test_columns = test.columns.to_list()
  miss_col = []
  for n in train_columns:
    if n not in test_columns:
      miss_col.append(n)
  return miss_col
test_not_filled_columns = searching_miss_col(train_not_filled, test_not_filled)
test_filled_columns = searching_miss_col(train_filled, test_filled)


def adding_miss_col(test, miss_columns):
  for col in miss_columns:
    test[col] = 0
  return test
adding_test_not_filled = adding_miss_col(test_not_filled, test_not_filled_columns)
adding_test_filled = adding_miss_col(test_filled, test_filled_columns)
#ahora tienen la misma cantidad de features, comprobable en cmpr1_filled y cmpr1_not_filled

def alineando(train, test):
  train = train.drop("SalePrice", axis=1)
  train_columns = train.columns.to_list()
  test = test[train_columns]
  return test
alineado_filled = alineando(train_filled, test_filled)
alineado_not_filled = alineando(train_not_filled, test_not_filled)
#ahora estan en el mismo orden los features, comprobable en cmpr2_filled, cmpr2_not_filled, cmpr3_filled y cmpr3_not_filled


def comprobacion1(train, test):
  comprob = set(train.columns) - {"SalePrice"} == set(test.columns)
  return comprob
cmpr1_not_filled = comprobacion1(train_not_filled, test_not_filled)
cmpr1_filled = comprobacion1(train_filled, test_filled)

def comprobacion2(alineado, train): #comprobación
  comprob = alineado.columns.equals(train.drop("SalePrice", axis=1).columns)
  return comprob
cmpr2_not_filled = comprobacion2(alineado_not_filled, train_not_filled)
cmpr2_filled = comprobacion2(alineado_filled, train_filled)

def comprobacion3(alineado, train):
  comprob = (train.drop("SalePrice", axis=1).columns == alineado.columns).all()
  return comprob
cmpr3_not_filled = comprobacion3(alineado_not_filled, train_not_filled)
cmpr3_filled = comprobacion3(alineado_filled, train_filled)

print(f"data not filled: {cmpr1_not_filled, cmpr2_not_filled, cmpr3_not_filled} \n data filled:{cmpr1_filled, cmpr2_filled, cmpr3_filled}")
#data filled:(True, True, np.True_)
#data not filled: (True, True, np.True_)


data not filled: (True, True, np.True_) 
 data filled:(True, True, np.True_)
